# StylePops — FashionCLIP LoRA Eğitimi (GitHub)

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Colab **Secrets** → `HF_TOKEN` = huggingface token
3. Hücreleri sırayla çalıştır

Repo: https://github.com/valueisinvalid/StylePops_Modules

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime → GPU (T4) seç!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torch torchvision transformers accelerate peft datasets pillow
!pip install -q 'torchao>=0.16.0'

In [ ]:
import os
from google.colab import userdata

# Colab Secrets: HF_TOKEN (önerilen)
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token: Colab Secrets ✓')
except Exception:
    # Manuel (tokenını buraya yapıştır)
    HF_TOKEN = ''  # hf_...
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
        print('HF token: manuel ✓')
    else:
        print('⚠️ HF_TOKEN yok — Secrets ekle veya HF_TOKEN doldur')

In [ ]:
import os, sys, shutil, subprocess

REPO = 'https://github.com/valueisinvalid/StylePops_Modules.git'
PROJECT_ROOT = '/content/StylePops_Modules'

# Önce güvenli dizine geç (rm -rf sonrası getcwd hatasını önler)
os.chdir('/content')

if os.path.isdir(PROJECT_ROOT):
    shutil.rmtree(PROJECT_ROOT)
    print('Eski kopya silindi')

subprocess.run(['git', 'clone', REPO, PROJECT_ROOT], check=True, cwd='/content')
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=PROJECT_ROOT, check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))
print('Proje:', os.getcwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip())

## Eğitim

**İlk deneme:** 3000 örnek (~15 dk)  
**Başarılıysa:** tam 44K hücresinin yorumunu kaldır

In [ ]:
import os
assert os.getcwd().endswith('StylePops_Modules'), 'Önce clone hücresini çalıştır'
!python scripts/train_fashionclip_lora.py --source hf --max-samples 3000 --epochs 1 --batch-size 16 --log-every 50

In [ ]:
# TAM EĞİTİM (test OK ise yorumu kaldır)
# !python scripts/train_fashionclip_lora.py --source hf --epochs 1 --batch-size 16 --log-every 100

In [ ]:
!ls -la outputs/fashionclip_lora/ 2>/dev/null || echo 'HATA: LoRA çıktısı yok'

In [ ]:
import os, shutil
from google.colab import files

lora = 'outputs/fashionclip_lora'
if os.path.exists(f'{lora}/adapter_config.json'):
    shutil.make_archive('/content/fashionclip_lora', 'zip', lora)
    files.download('/content/fashionclip_lora.zip')
    print('İndirildi ✓')
else:
    print('HATA: adapter_config.json yok — eğitim tamamlanmadı')

## Mac'te

1. `fashionclip_lora.zip` → `StylePops_Modules/outputs/fashionclip_lora/`
2. `python scripts/train_compatibility_head.py`